<a href="https://colab.research.google.com/github/MarkDeng86/VolatilityPredictionDeepLearning/blob/GARCH/%E2%80%9COptiverRV_ipynb%E2%80%9D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# @title
# importing kaggle competition dataset
# make sure to upload kaggle.json to google drive
competition_name = "optiver-realized-volatility-prediction"

# Mount your Google Drive.
from google.colab import drive
drive.mount("/content/drive")

kaggle_creds_path = "/content/drive/MyDrive/kaggle.json"

! pip install kaggle --quiet
! cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
! mkdir ~/.kaggle

! kaggle competitions download -c {competition_name}

! mkdir kaggle_data
! unzip {competition_name + ".zip"} -d kaggle_data

# Unmount your Google Drive
drive.flush_and_unmount()

### IGNORE THIS CELL

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: cannot stat '/content/drive/MyDrive/kaggle.json': No such file or directory
mkdir: cannot create directory ‘/root/.kaggle’: File exists
You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
mkdir: cannot create directory ‘kaggle_data’: File exists
unzip:  cannot find or open optiver-realized-volatility-prediction.zip, optiver-realized-volatility-prediction.zip.zip or optiver-realized-volatility-prediction.zip.ZIP.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(2026)
DATA_DIR = '/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/kaggle_data/optiver-realized-volatility-prediction'

### Mark: Below I am testing implementation with polars for more efficient data manipulation. We can translate pandas manipulation to polars pretty easily.

In [5]:
import polars as pl

In [6]:
pl_lazy_book_train = pl.scan_parquet(f'{DATA_DIR}/book_train.parquet', hive_partitioning=True)

In [7]:
# checking book data schema
check_pl = pl.read_parquet_schema(f'{DATA_DIR}/book_train.parquet/stock_id=0/c439ef22282f412ba39e9137a3fdabac.parquet')
print(check_pl)

Schema({'time_id': Int16, 'seconds_in_bucket': Int16, 'bid_price1': Float32, 'ask_price1': Float32, 'bid_price2': Float32, 'ask_price2': Float32, 'bid_size1': Int32, 'ask_size1': Int32, 'bid_size2': Int32, 'ask_size2': Int32})


In [8]:
# TRANSFORM: Define your preprocessing steps
# Polars builds an optimized query plan for these operations
processed_book = (
    pl_lazy_book_train
    # OPTIMIZATION: Convert stock_id to Int8 early to save memory
    .with_columns(pl.col("stock_id").cast(pl.Int8))
    # Ensure correct ordering for time-based calculations
    .sort(["stock_id", "time_id", "seconds_in_bucket"])

    .with_columns(
        # Calculate the primary WAP (Weighted Average Price)
        wap1 = ((pl.col("bid_price1") * pl.col("ask_size1") +
                pl.col("ask_price1") * pl.col("bid_size1")) /
                (pl.col("bid_size1") + pl.col("ask_size1"))).cast(pl.Float32),

        wap2 = ((pl.col("bid_price2") * pl.col("ask_size2") +
                pl.col("ask_price2") * pl.col("bid_size2")) /
                (pl.col("bid_size2") + pl.col("ask_size2"))).cast(pl.Float32),

        # Calculate the level 1 Bid-Ask spread
        spread1 = (pl.col("ask_price1") - pl.col("bid_price1")).cast(pl.Float32),

        # Calculate the level 1 and level 1&2 volume
        vol_l1 = (pl.col("bid_size1") + pl.col("ask_size1")),
        vol_l2 = (pl.col("bid_size2") + pl.col("ask_size2"))
    )
    .with_columns(
        # Calculate the total volume
        vol_total = (pl.col("vol_l1") + pl.col("vol_l2")),

        # Calculate the volume imbalance -> Float32
        vol_imbalance = ((pl.col("bid_size1") - pl.col("ask_size1")) /
                        (pl.col("bid_size1") + pl.col("ask_size1"))).cast(pl.Float32),

        wap_basis = (pl.col("wap1") - pl.col("wap2")).cast(pl.Float32)
    )

    # Time-Based Features
    .with_columns(
        # Create 60-second micro-buckets
        minute_in_bucket = (pl.col("seconds_in_bucket") // 60).cast(pl.Int8),

        # Calculate time elapsed since last update
        time_since_last_update = (
            pl.col("seconds_in_bucket") - pl.col("seconds_in_bucket")
            .shift(1)
            .over(["stock_id", "time_id"])
        ).fill_null(0).cast(pl.Int16)
    )

    # Lagged Variables
    .with_columns(
        # Tick-to-tick Log Return of WAP
        log_return_1t = (
            pl.col("wap1").log() - pl.col("wap1").shift(1).log()
        ).over(["stock_id", "time_id"]).fill_null(0).cast(pl.Float32),

        # Spread change
        spread_change = (
            pl.col("spread1") - pl.col("spread1").shift(1)
        ).over(["stock_id", "time_id"]).fill_null(0).cast(pl.Float32)
    )
    .with_columns(
        # Squared returns
        log_return_sq = (pl.col("log_return_1t") ** 2).cast(pl.Float32)
    )
    .with_columns(
        # Realized volatility
        realized_volatility = pl.col("log_return_sq").sum().sqrt().over(["stock_id", "time_id"]).cast(pl.Float32)
    )

    # Rolling window statistics
    .with_columns(
        # 10-tick rolling mean of spread
        spread_rolling_mean_10t = (
            pl.col("spread1")
            .rolling_mean(window_size=10, min_samples=1) # min_samples overides min tick count require for 10t statistics; addressing null values
            .over(["stock_id", "time_id"])
        ).cast(pl.Float32),

        # 20-tick rolling std of log returns
        volatility_rolling_20t = (
            pl.col("log_return_1t")
            .rolling_std(window_size=20, min_samples=2)
            .over(["stock_id", "time_id"])
        ).cast(pl.Float32),

        # Moving sum of order flow imbalance
        order_flow_imbalance_15t = (
            (pl.col("bid_size1") - pl.col("ask_size1"))
            .rolling_sum(window_size=15, min_samples=1)
            .over(["stock_id", "time_id"])
        ).cast(pl.Int32)
    )
)

In [9]:
# Group by stock and time, then calculate summary statistics (e.g., standard deviation of WAP)
aggregated_features = (
    processed_book
    .group_by(["stock_id", "time_id"])
    .agg(
        # Example feature: Realized volatility (std dev of log returns or WAP variations)
        realized_volatility = pl.col("log_return_sq").sum().sqrt(),
        wap1_mean = pl.col("wap1").mean(),

        # Average spread over the time bucket
        mean_spread = pl.col("spread1").mean(),

        # Count the number of updates in this bucket
        update_count = pl.col("seconds_in_bucket").count(),

        # ---- Volume Features ----
        # vol_mean = pl.col("vol_total").mean(),
        # vol_std = pl.col("vol_total").std(),
        vol_sum = pl.col("vol_total").sum(),

        vol_90 = pl.col("vol_total").quantile(0.9),
        # vol_99 = pl.col("vol_total").quantile(0.99),

        vol_imbalance_mean = pl.col("vol_imbalance").mean(),
        vol_imbalance_std = pl.col("vol_imbalance").std()
    )
)

In [10]:
# I can't run the code below

In [ ]:
processed_book.fetch(5)

/tmp/ipykernel_5778/3337148965.py:1: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  processed_book.fetch(5)


In [ ]:
processed_book_df = processed_book.limit(5).collect()

print(processed_book_df.head())

In [ ]:
with pl.Config(tbl_cols=-1): # Set context to show all columns
    # check null values
    null_count_df = processed_book_df.null_count()
    print(null_count_df)

In [ ]:
# Finally, execute the lazy computation and load the aggregated result into memory
aggregated_df = aggregated_features.collect()

print(aggregated_df.head())

In [ ]:
df_sample = (
    processed_book
    .select(["spread1", "vol_total", "vol_imbalance"])
    .collect()
    .sample(fraction=0.01, seed=42) # Adjust fraction based on your RAM
)

# Plotting the global distribution of the Order Book Imbalance
plt.figure(figsize=(10, 5))
sns.histplot(df_sample['vol_imbalance'].to_numpy(), bins=50, kde=True, color='purple')
plt.title('Global Distribution of Order Book Imbalance (1% Sample)')
plt.xlabel('Order Book Imbalance (-1 to 1)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Spearman rank correlation Heat map
# Exclude identifiers (we don't want to correlate stock_id or time_id)
df_numeric = aggregated_features.drop(["stock_id", "time_id"])

# Convert to Pandas and calculate the Spearman correlation matrix
# Seaborn natively understands Pandas DataFrames best for index/column labeling
corr_matrix = (df_numeric.collect()).to_pandas().corr(method='spearman')

# Visualize the Heatmap
plt.figure(figsize=(12, 10))

# Create the heatmap
sns.heatmap(
    corr_matrix,
    annot=True,          # Display the correlation coefficients
    fmt=".2f",           # Round to 2 decimal places
    cmap="coolwarm",     # Red for positive correlation, Blue for negative
    vmin=-1, vmax=1,     # Fix the scale from -1 to 1
    center=0,            # Center the colormap at 0
    square=True,         # Make the grid cells square
    linewidths=.5,       # Add lines between cells for readability
    cbar_kws={"shrink": .8} # Shrink the colorbar slightly
)

plt.title("Spearman Correlation Heatmap of Aggregated LOB Features", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

#### Time Series Based Feature Engineering with Polars
Before applying any time-dependent transformations, you must ensure the data is strictly sorted chronologically. If the rows are out of order, your lags and rolling windows will be corrupted.

#### Time-Based Features
Because `seconds_in_bucket` resets to 0 for every `time_id`, it acts as our temporal axis. We can derive features that help a Transformer understand where it is within the 600-second window.

In [ ]:
time_features = processed_book.with_columns(
    # Create 60-second micro-buckets (values 0 through 9)
    # This helps models capture structural behaviors, like end-of-bucket trading spikes
    minute_in_bucket = (pl.col("seconds_in_bucket") // 60).cast(pl.Int8),

    # Calculate the time elapsed since the last order book update
    # We fill the first null value in each bucket with 0
    time_since_last_update = (
        pl.col("seconds_in_bucket")
        - pl.col("seconds_in_bucket").shift(1)
    ).over(["stock_id", "time_id"]).fill_null(0)
)

 #### Lagged Variables
Lagging allows the model to look at the recent past to calculate differentials, such as returns or changes in the bid-ask spread. We use `.shift()` combined with `.over()` to enforce our boundaries.

In [ ]:
lagged_features = (
      time_features.with_columns(
      # Calculate the tick-to-tick Log Return of the Weighted Average Price
      log_return_1t = (
          pl.col("wap1").log() - pl.col("wap1").shift(1).log()
      ).over(["stock_id", "time_id"]),

      # Calculate how much the spread widened or narrowed since the last update
      spread_change = (
          pl.col("spread1") - pl.col("spread1").shift(1)
      ).over(["stock_id", "time_id"])
  ).fill_null(0) # The first return in a bucket is 0
  .with_columns(
        # Calculate squared returns
        log_return_sq=pl.col("log_return") ** 2
    )
    .with_columns(
        # Calculate expanding realized volatility
        realized_volatility=pl.col("log_return_sq").cum_sum().sqrt().over("time_id")
    )
)

#### Moving Window Statistics
Because order book updates arrive irregularly, you have two choices for rolling windows: tick-based (number of rows) or time-based (number of seconds). <br>
Here is how to calculate tick-based rolling statistics using `.rolling_mean()` and `.rolling_std()`. These will calculate metrics over the last $N$ updates, regardless of how much time passed between them.

In [ ]:
rolling_features = lagged_features.with_columns(
    # 10-tick rolling mean of the bid-ask spread
    spread_rolling_mean_10t = (
        pl.col("spread1")
        .rolling_mean(window_size=10)
        .over(["stock_id", "time_id"])
    ),

    # 20-tick rolling standard deviation of log returns (Realized Micro-Volatility)
    # We use a min_periods of 2 so it starts calculating before reaching 20 ticks
    volatility_rolling_20t = (
        pl.col("log_return_1t")
        .rolling_std(window_size=20)
        .over(["stock_id", "time_id"])
    ),

    # Moving sum of order flow imbalance over the last 15 updates
    order_flow_imbalance_15t = (
        (pl.col("bid_size1") - pl.col("ask_size1"))
        .rolling_sum(window_size=15)
        .over(["stock_id", "time_id"])
    )
)


In [ ]:
# taking a look at a particular stock
target_stock = 0
target_time_id = 100

df_subset_book = (
    processed_book_df
    .filter((pl.col("stock_id") == target_stock) & (pl.col("time_id") < target_time_id))
)
df_subset_book

In [ ]:
df_trade = pl.scan_parquet(f'{DATA_DIR}/trade_train.parquet', hive_partitioning=True)

df_trade_adv = (
    df_trade
    .sort(["stock_id", "time_id", "seconds_in_bucket"])
    .with_columns(
        # 1. Inter-Arrival Time (Seconds since the last trade in this bucket)
        inter_arrival_time=pl.col("seconds_in_bucket").diff().over(["stock_id", "time_id"]).fill_null(0),

        # 2. Average Trade Size (Institutional vs. Retail proxy)
        avg_trade_size=(pl.col("size") / pl.col("order_count"))
    )
)
trade_df = df_trade_adv.collect()
trade_df

In [ ]:
target_stock = 5
target_time_id = 100

df_subset_trade = (
    trade_df
    .filter((pl.col("stock_id") == target_stock) & (pl.col("time_id") < target_time_id))
)
df_subset_trade

In [ ]:
import hvplot.polars
df_subset_trade.hvplot.scatter(
    x="seconds_in_bucket",
    y="price",
    by="time_id",
    width=650,
    title="Executed Trade",
    xlabel='seconds_in_bucket',
    ylabel='price',
)

In [ ]:
# loading the whole dataset
# df_book_train_raw = pd.read_parquet(f'{DATA_DIR}/book_train.parquet')
# df_book_test_raw = pd.read_parquet(f'{DATA_DIR}/book_test.parquet')

# df_trade_train_raw = pd.read_parquet(f'{DATA_DIR}/trade_train.parquet')
# df_trade_test_raw = pd.read_parquet(f'{DATA_DIR}/trade_test.parquet')

df_train_raw = pd.read_csv(f'{DATA_DIR}/train.csv')
df_test_raw = pd.read_csv(f'{DATA_DIR}/test.csv')

In [ ]:
# display(df_train_raw.head(5))
# display(df_test_raw.head(5))
display(df_book_train_raw.head(5))
display(df_trade_train_raw.head(5))

In [ ]:
# loading order book data by single stock
df_book_train_stock_id_0 = pd.read_parquet(f'{DATA_DIR}/book_train.parquet/stock_id=0')
display(df_book_train_stock_id_0.head(5))

We can start by implementing the EDA functions on a single stock for now

In [ ]:
df_book_train_stock_id_0.isna().sum()

In [ ]:
df_book_train_stock_id_0['bid_volume'] = df_book_train_stock_id_0[['bid_size1', 'bid_size2']].sum(axis=1)
df_book_train_stock_id_0['ask_volume'] = df_book_train_stock_id_0[['ask_price1', 'ask_price2']].sum(axis=1)
df_book_train_stock_id_0['total_volume'] = df_book_train_stock_id_0['bid_volume'] + df_book_train_stock_id_0['ask_volume']

In [ ]:
df_book_train_stock_id_0[['bid_volume', 'ask_volume', 'total_volume']].describe()

In [ ]:
df_book_train_stock_id_0.groupby('time_id')['total_volume'].describe()

This part is a standard way of implementing a GARCH

In [ ]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox

from arch import arch_model

In [ ]:
# Convert price to returns
# Assume df has a 'price' column and is time-ordered
df = df.sort_values('time')

df['log_return'] = np.log(df['price']).diff()
returns = df['log_return'].dropna()

# Scale
returns = returns * 100

In [ ]:
# visual plot
plt.figure(figsize=(10,4))
plt.plot(returns)
plt.title("Returns")
plt.show()

In [ ]:
# Check stationarity
adf_stat, p_value, _, _, _, _ = adfuller(returns)

print("ADF Statistic:", adf_stat)
print("p-value:", p_value)

In [ ]:
# acf, pacf
plot_acf(returns, lags=30)
plot_pacf(returns, lags=30)
plt.show()

In [ ]:
# arch effect test
arch_stat, arch_pvalue, _, _ = het_arch(returns)

print("ARCH test p-value:", arch_pvalue)

In [ ]:
# ACF of squared returns
plot_acf(returns**2, lags=30)
plt.title("ACF of Squared Returns")
plt.show()

In [ ]:
# fit in the model
model = arch_model(
    returns,
    mean='Zero',
    vol='Garch',
    p=1,
    q=1,
    dist='t'
)

res = model.fit(disp='off')
print(res.summary())

In [ ]:
# output
cond_vol = res.conditional_volatility
residuals = res.resid
std_resid = residuals / cond_vol

In [ ]:
# residual autocorrelation
plot_acf(std_resid, lags=30)
plt.title("ACF of Standardized Residuals")
plt.show()

In [ ]:
# squared residuals
plot_acf(std_resid**2, lags=30)
plt.title("ACF of Squared Standardized Residuals")
plt.show()

In [ ]:
# L-B test
lb_test = acorr_ljungbox(std_resid, lags=[10], return_df=True)
print(lb_test)

In [ ]:
# arch test
arch_stat2, arch_pvalue2, _, _ = het_arch(std_resid)
print("Post-GARCH ARCH p-value:", arch_pvalue2)

In [ ]:
# forecaste
forecast = res.forecast(horizon=1)

next_variance = forecast.variance.iloc[-1, 0]
next_volatility = np.sqrt(next_variance)

print("Next period volatility:", next_volatility)